In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        (os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [2]:
pip install torch torchvision timm opencv-python scikit-learn matplotlib


Note: you may need to restart the kernel to use updated packages.


In [3]:
import os
import cv2
import torch
import timm
import numpy as np
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from sklearn.metrics import classification_report, confusion_matrix


/usr/local/lib/python3.12/dist-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'repr' attribute with value False was provided to the `Field()` function, which has no effect in the context it was used. 'repr' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` statement was used, or if the `Field()` function was attached to a single member of a union type.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'frozen' attribute with value True was provided to the `Field()` function, which has no effect in the context it was used. 'frozen' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` 

In [4]:
class MRIDataset(Dataset):
    def __init__(self, root_dir, transform=None, max_slices=16):
        self.samples = []
        self.transform = transform
        self.max_slices = max_slices
        self.class_map = {cls:i for i,cls in enumerate(sorted(os.listdir(root_dir)))}

        for cls in self.class_map:
            cls_path = os.path.join(root_dir, cls)
            for case in os.listdir(cls_path):
                self.samples.append((os.path.join(cls_path, case), self.class_map[cls]))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        case_path, label = self.samples[idx]
        slices = sorted(os.listdir(case_path))[:self.max_slices]

        imgs = []
        for s in slices:
            img = cv2.imread(os.path.join(case_path, s))
            img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
            if self.transform:
                img = self.transform(img)
            imgs.append(img)

        imgs = torch.stack(imgs)
        return imgs, label


In [5]:
transform = transforms.Compose([
    transforms.ToPILImage(),
    transforms.Resize((224,224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5]*3, std=[0.5]*3)
])


In [6]:
class TemporalAttention(nn.Module):
    def __init__(self, hidden_dim):
        super().__init__()
        self.attn = nn.Linear(hidden_dim, 1)

    def forward(self, x):
        weights = torch.softmax(self.attn(x), dim=1)
        return (weights * x).sum(dim=1)


In [7]:
class Swin_BiLSTM(nn.Module):
    def __init__(self, num_classes=4):
        super().__init__()

        self.swin = timm.create_model(
            "swin_tiny_patch4_window7_224",
            pretrained=True,
            num_classes=0
        )

        self.lstm = nn.LSTM(
            input_size=768,
            hidden_size=256,
            bidirectional=True,
            batch_first=True
        )

        self.attention = TemporalAttention(512)

        self.classifier = nn.Sequential(
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(256, num_classes)
        )

    def forward(self, x):
        B, T, C, H, W = x.shape
        x = x.view(B*T, C, H, W)

        feats = self.swin(x)
        feats = feats.view(B, T, -1)

        lstm_out, _ = self.lstm(feats)
        attn_out = self.attention(lstm_out)

        return self.classifier(attn_out)


In [8]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

train_ds = MRIDataset("/kaggle/input/brain-tumor-mri-dataset/Training", transform)
val_ds   = MRIDataset("/kaggle/input/brain-tumor-mri-dataset/Testing", transform)

train_loader = DataLoader(train_ds, batch_size=2, shuffle=True)
val_loader   = DataLoader(val_ds, batch_size=2)

model = Swin_BiLSTM(num_classes=4).to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4)


model.safetensors:   0%|          | 0.00/114M [00:00<?, ?B/s]

In [9]:
for epoch in range(20):
    model.train()
    total_loss = 0

    for x, y in train_loader:
        x, y = x.to(device), y.to(device)

        optimizer.zero_grad()
        preds = model(x)
        loss = criterion(preds, y)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print(f"Epoch {epoch+1} | Loss: {total_loss/len(train_loader):.4f}")


NotADirectoryError: [Errno 20] Not a directory: '/kaggle/input/brain-tumor-mri-dataset/Training/glioma/Tr-gl_0391.jpg'

In [ ]:
model.eval()
y_true, y_pred = [], []

with torch.no_grad():
    for x, y in val_loader:
        x = x.to(device)
        preds = model(x).argmax(1).cpu().numpy()
        y_true.extend(y.numpy())
        y_pred.extend(preds)

print(classification_report(y_true, y_pred))
print(confusion_matrix(y_true, y_pred))
